# 01. Data Loading and Cleaning Pipeline — EMIPredict AI

> **Environment Note:** The pip install command below pins exact ML library versions for the Google Colab / Jupyter notebook environment. The deployed full-stack Node.js application uses native ONNX inference and does not install from this file.

In [1]:
# Pin dependencies for Google Colab environment
%pip install -r ../requirements.txt if os.path.exists('../requirements.txt') else %pip install -r requirements.txt

/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `pip install -r ../requirements.txt if os.path.exists('../requirements.txt') else %pip install -r requirements.txt'


In [2]:
import os
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load raw dataset
possible_paths = [
    '../data/raw/EMI_dataset.csv',
    'data/raw/EMI_dataset.csv',
    '../data/raw/emi_prediction_dataset.csv',
    'data/raw/emi_prediction_dataset.csv'
]

raw_data_path = next((p for p in possible_paths if os.path.exists(p)), 'data/raw/EMI_dataset.csv')
df = pd.read_csv(raw_data_path)
print(f"Raw Dataset Loaded from '{raw_data_path}': {df.shape[0]:,} rows, {df.shape[1]} columns.")

/tmp/ipykernel_12337/4046955362.py:16: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(raw_data_path)


Raw Dataset Loaded from '../data/raw/EMI_dataset.csv': 404,800 rows, 27 columns.


In [3]:
# 1.1 Fix Malformed Numeric Columns (age, monthly_salary, bank_balance)
# Addresses duplicated decimal suffixes like '64300.0.0' or '58.0.0'
malformed_cols = ['age', 'monthly_salary', 'bank_balance']
num_pattern = re.compile(r'^(-?\d+\.?\d*)')

print("=== REPAIRING MALFORMED NUMERIC COLUMNS ===")
for col in malformed_cols:
    if col in df.columns:
        orig_nulls = df[col].isnull().sum()
        if not pd.api.types.is_numeric_dtype(df[col]):
            # Extract first valid numeric substring
            cleaned_series = df[col].astype(str).str.extract(r'^(-?\d+\.?\d*)')[0]
            # Coerce to numeric float
            numeric_col = pd.to_numeric(cleaned_series, errors='coerce')
            
            # Count modified and coerced NaNs
            malformed_count = (df[col].astype(str) != cleaned_series).sum()
            new_nulls = numeric_col.isnull().sum() - orig_nulls
            df[col] = numeric_col
            print(f"Column '{col}': Converted to numeric float ({malformed_count:,} malformed values repaired, {new_nulls:,} unparseable coerced to NaN)")
        else:
            print(f"Column '{col}': Already numeric dtype ({df[col].dtype})")

=== REPAIRING MALFORMED NUMERIC COLUMNS ===


Column 'age': Converted to numeric float (3 malformed values repaired, 0 unparseable coerced to NaN)


Column 'monthly_salary': Converted to numeric float (1,993 malformed values repaired, 0 unparseable coerced to NaN)


Column 'bank_balance': Converted to numeric float (4,392 malformed values repaired, 14 unparseable coerced to NaN)


In [4]:
# 1.2 Normalize Gender to 2 Canonical Categories ('Female', 'Male')
print("\n=== GENDER NORMALIZATION ===")
print("Value counts before:")
print(df['gender'].value_counts(dropna=False))

def normalize_gender(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    if s.startswith('f'):
        return 'Female'
    elif s.startswith('m'):
        return 'Male'
    return val

df['gender'] = df['gender'].apply(normalize_gender)
print("\nValue counts after normalization:")
print(df['gender'].value_counts(dropna=False))

# 1.3 Strip ' EMI' suffix from emi_scenario
print("\n=== EMI SCENARIO NORMALIZATION ===")
print("Scenarios before cleaning:", df['emi_scenario'].dropna().unique())
df['emi_scenario'] = df['emi_scenario'].astype(str).str.replace(r'\s+EMI$', '', regex=True).str.strip()
expected_scenarios = {'E-commerce Shopping', 'Education', 'Home Appliances', 'Personal Loan', 'Vehicle'}
actual_scenarios = set(df['emi_scenario'].dropna().unique())
print("Scenarios after cleaning:", actual_scenarios)
assert actual_scenarios == expected_scenarios, f"Mismatch in emi_scenario categories! Expected {expected_scenarios}, got {actual_scenarios}"

# 1.4 Convert existing_loans from Yes/No Text to 0/1 Numeric Integer
print("\n=== EXISTING LOANS CONVERSION ===")
print("Value counts before:")
print(df['existing_loans'].value_counts(dropna=False))
loan_mapping = {'yes': 1, 'true': 1, '1': 1, '1.0': 1, 'no': 0, 'false': 0, '0': 0, '0.0': 0}
df['existing_loans'] = df['existing_loans'].astype(str).str.strip().str.lower().map(loan_mapping).fillna(0).astype(int)
print("\nValue counts after mapping to 0/1:")
print(df['existing_loans'].value_counts(dropna=False))


=== GENDER NORMALIZATION ===
Value counts before:
gender
Male      237427
Female    158351
MALE        1865
M           1843
male        1815
F           1171
female      1165
FEMALE      1163
Name: count, dtype: int64



Value counts after normalization:
gender
Male      242950
Female    161850
Name: count, dtype: int64

=== EMI SCENARIO NORMALIZATION ===
Scenarios before cleaning: ['Personal Loan EMI' 'E-commerce Shopping EMI' 'Education EMI'
 'Vehicle EMI' 'Home Appliances EMI']


Scenarios after cleaning: {'Home Appliances', 'Vehicle', 'E-commerce Shopping', 'Personal Loan', 'Education'}

=== EXISTING LOANS CONVERSION ===
Value counts before:
existing_loans
No     243227
Yes    161573
Name: count, dtype: int64

Value counts after mapping to 0/1:
existing_loans
0    243227
1    161573
Name: count, dtype: int64


In [5]:
# 2. Data Audit: Nulls, Duplicates & Datatypes
print("=== DATA AUDIT SUMMARY ===")
print(df.info())
print("\nMissing Values per Column before Imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

duplicate_count = df.duplicated().sum()
print(f"\nExact Duplicate Rows Found: {duplicate_count}")
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Removed exact duplicates. Shape after drop: {df.shape}")

=== DATA AUDIT SUMMARY ===


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404800 entries, 0 to 404799
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   age                     404800 non-null  float64
 1   gender                  404800 non-null  object 
 2   marital_status          404800 non-null  object 
 3   education               402396 non-null  object 
 4   monthly_salary          404800 non-null  float64
 5   employment_type         404800 non-null  object 
 6   years_of_employment     404800 non-null  float64
 7   company_type            404800 non-null  object 
 8   house_type              404800 non-null  object 
 9   monthly_rent            402374 non-null  float64
 10  family_size             404800 non-null  int64  
 11  dependents              404800 non-null  int64  
 12  school_fees             404800 non-null  float64
 13  college_fees            404800 non-null  float64
 14  travel_expenses     

education         2404
monthly_rent      2426
credit_score      2420
bank_balance      2440
emergency_fund    2351
dtype: int64



Exact Duplicate Rows Found: 0


In [6]:
# 1.5 Null Value Imputation (Median for Numeric, Mode for Categorical)
# Note: age, monthly_salary, bank_balance are now numeric float columns,
# so they correctly receive median imputation along with monthly_rent, credit_score, and emergency_fund.
print("=== NULL VALUE IMPUTATION ===")
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"[NUMERIC] Imputed missing values in '{col}' with median: {median_val}")

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"[CATEGORICAL] Imputed missing values in '{col}' with mode: {mode_val}")

# Strict assertion: confirm 0 nulls remain across all columns
remaining_nulls = df.isnull().sum().sum()
assert remaining_nulls == 0, f"Assertion Failed: Found {remaining_nulls} unhandled nulls: {df.isnull().sum().to_dict()}"
print(f"\n✓ Null imputation complete: 0 missing values remain across all {df.shape[1]} columns.")

=== NULL VALUE IMPUTATION ===


[NUMERIC] Imputed missing values in 'monthly_rent' with median: 0.0
[NUMERIC] Imputed missing values in 'credit_score' with median: 701.0
[NUMERIC] Imputed missing values in 'bank_balance' with median: 196000.0
[NUMERIC] Imputed missing values in 'emergency_fund' with median: 74000.0
[CATEGORICAL] Imputed missing values in 'education' with mode: Graduate



✓ Null imputation complete: 0 missing values remain across all 27 columns.


In [7]:
# 4. Outlier Flagging via IQR (Non-destructive)
def flag_iqr_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outlier_mask = (data[column] < lower_bound) | (data[column] > upper_bound)
    outlier_cnt = outlier_mask.sum()
    print(f"Column '{column}': {outlier_cnt:,} outliers flagged ({outlier_cnt / len(data) * 100:.2f}%) [Bounds: {lower_bound:.1f} to {upper_bound:.1f}]")
    return outlier_mask

print("=== OUTLIER AUDIT (IQR) ===")
df['outlier_flag_salary'] = flag_iqr_outliers(df, 'monthly_salary')
df['outlier_flag_credit'] = flag_iqr_outliers(df, 'credit_score')
df['outlier_flag_balance'] = flag_iqr_outliers(df, 'bank_balance')
# Note: Outliers are flagged for audit but retained to preserve valid high-income and high-net-worth applicant profiles.

=== OUTLIER AUDIT (IQR) ===
Column 'monthly_salary': 12,120 outliers flagged (2.99%) [Bounds: -21000.0 to 129400.0]
Column 'credit_score': 5,840 outliers flagged (1.44%) [Bounds: 513.0 to 889.0]
Column 'bank_balance': 13,372 outliers flagged (3.30%) [Bounds: -232750.0 to 667650.0]


In [8]:
# 2.1 Final Schema and Data Quality Sanity Checks
print("=== FINAL VERIFICATION & SANITY CHECKS ===")

# Check 1: Gender contains only canonical values
gender_vals = set(df['gender'].dropna().unique())
is_gender_valid = gender_vals.issubset({'Female', 'Male'})
print(f"[{'PASS' if is_gender_valid else 'FAIL'}] Gender unique values: {gender_vals}")
assert is_gender_valid, f"Gender check failed: {gender_vals}"

# Check 2: EMI Scenarios match 5 canonical names
scenario_vals = set(df['emi_scenario'].unique())
is_scenario_valid = scenario_vals == {'E-commerce Shopping', 'Education', 'Home Appliances', 'Personal Loan', 'Vehicle'}
print(f"[{'PASS' if is_scenario_valid else 'FAIL'}] EMI Scenario categories: {scenario_vals}")
assert is_scenario_valid, f"EMI scenario check failed: {scenario_vals}"

# Check 3: Existing loans contains only 0 and 1
loan_vals = set(df['existing_loans'].unique())
is_loan_valid = loan_vals.issubset({0, 1})
print(f"[{'PASS' if is_loan_valid else 'FAIL'}] Existing loans unique values: {loan_vals}")
assert is_loan_valid, f"Existing loans check failed: {loan_vals}"

# Check 4: age, monthly_salary, bank_balance are numeric dtypes
is_age_num = pd.api.types.is_numeric_dtype(df['age'])
is_sal_num = pd.api.types.is_numeric_dtype(df['monthly_salary'])
is_bal_num = pd.api.types.is_numeric_dtype(df['bank_balance'])
print(f"[{'PASS' if is_age_num else 'FAIL'}] 'age' is numeric dtype: {df['age'].dtype}")
print(f"[{'PASS' if is_sal_num else 'FAIL'}] 'monthly_salary' is numeric dtype: {df['monthly_salary'].dtype}")
print(f"[{'PASS' if is_bal_num else 'FAIL'}] 'bank_balance' is numeric dtype: {df['bank_balance'].dtype}")
assert is_age_num and is_sal_num and is_bal_num, "Numeric column type assertion failed!"

print("\n=======================================================")
print("ALL DATA CLEANING & SCHEMA SANITY CHECKS PASSED (100%)")
print("=======================================================")

=== FINAL VERIFICATION & SANITY CHECKS ===
[PASS] Gender unique values: {'Female', 'Male'}
[PASS] EMI Scenario categories: {'Home Appliances', 'Vehicle', 'E-commerce Shopping', 'Personal Loan', 'Education'}
[PASS] Existing loans unique values: {0, 1}
[PASS] 'age' is numeric dtype: float64
[PASS] 'monthly_salary' is numeric dtype: float64
[PASS] 'bank_balance' is numeric dtype: float64

ALL DATA CLEANING & SCHEMA SANITY CHECKS PASSED (100%)


In [9]:
# 5. Stratified 70/15/15 Train / Validation / Test Split
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['emi_eligibility']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['emi_eligibility']
)

df['dataset_split'] = 'train'
df.loc[val_df.index, 'dataset_split'] = 'val'
df.loc[test_df.index, 'dataset_split'] = 'test'

print("\n=== STRATIFIED SPLIT DISTRIBUTION ===")
print(df['dataset_split'].value_counts(normalize=True))

# 6. Save Cleaned Dataset
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
output_file = '../data/processed/cleaned_dataset.csv' if os.path.exists('../data') else 'data/processed/cleaned_dataset.csv'
df.to_csv(output_file, index=False)
print(f"\nCleaned dataset successfully saved to: {output_file}")


=== STRATIFIED SPLIT DISTRIBUTION ===
dataset_split
train    0.70
test     0.15
val      0.15
Name: proportion, dtype: float64



Cleaned dataset successfully saved to: ../data/processed/cleaned_dataset.csv


## Cleaning Decision Summary

- **Malformed Numeric Repair**: Extracted valid numeric prefixes from `age`, `monthly_salary`, and `bank_balance` before datatype conversion to avoid silent failure or improper mode imputation.
- **Categorical Normalization**: Normalized `gender` to exactly 2 canonical classes (`Female`, `Male`) and stripped the `' EMI'` suffix from `emi_scenario` to align with the 48-feature production schema.
- **Binary Conversion**: Mapped `existing_loans` from string representations to `0` and `1` integers.
- **Duplicates & Null Imputation**: Dropped exact duplicate rows; imputed numerical features using column median and categorical features using column mode. Verified 0 nulls remain.
- **Outlier Handling**: Evaluated `monthly_salary`, `credit_score`, and `bank_balance` using IQR thresholds. Outliers are flagged for auditing but retained to preserve valid financial extremes.
- **Splitting Strategy**: Performed a stratified 70/15/15 split on `emi_eligibility` to maintain target class distribution across train, validation, and test sets.